# Import Fine-tuned LLaMA 3 models on SageMaker JumpStart to private model hub (SageMaker Python SDK v3)

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

---
In this demo notebook, we demonstrate how to use the SageMaker Python SDK to deploy pre-trained Llama 3 model as well as fine-tune it for your dataset in domain adaptation or instruction tuning format. We will then import the model into a jumpstart private model hub.

---

### Model License information
---
To perform inference on these models, you need to pass custom_attributes='accept_eula=true' as part of header. This means you have read and accept the end-user-license-agreement (EULA) of the model. EULA can be found in model card description or from https://ai.meta.com/resources/models-and-libraries/llama-downloads/. By default, this notebook sets custom_attributes='accept_eula=false', so all inference requests will fail until you explicitly change this custom attribute.

Note: Custom_attributes used to pass EULA are key/value pairs. The key and value are separated by '=' and pairs are separated by ';'. If the user passes the same key more than once, the last value is kept and passed to the script handler (i.e., in this case, used for conditional logic). For example, if 'accept_eula=false; accept_eula=true' is passed to the server, then 'accept_eula=true' is kept and passed to the script handler.

---

### Set up

---
We begin by installing and upgrading necessary packages. Restart the kernel after executing the cell below for the first time.

---

In [ ]:
# SageMaker Python SDK v3 is required for this notebook. Install/upgrade it
# together with the datasets library used for dataset preparation.
!pip install --upgrade "sagemaker>=3.0" datasets

In [ ]:
import warnings
import logging


# Suppress warnings
warnings.filterwarnings('ignore')

# Suppress INFO messages
logging.getLogger().setLevel(logging.ERROR)

## Deploy Pre-trained Model

---

First we will deploy the Llama-2 model as a SageMaker endpoint. To train/deploy 8B and 70B models, please change model_id to "meta-textgeneration-llama-3-8b" and "meta-textgeneration-llama-3-70b" respectively.

---

In [ ]:
# Pin a full semantic version (not a wildcard like "2.*"). V3's
# ModelTrainer.from_jumpstart_config validates the JumpStart hub content version
# via describe_hub_content, which requires a full semver (min length 5); a
# wildcard would raise a ParamValidationError on HubContentVersion.
model_id, model_version = "meta-textgeneration-llama-3-8b", "2.9.0"

In [ ]:
# V3: deploy a JumpStart model with ModelBuilder.from_jumpstart_config.
# (V2 used sagemaker.jumpstart.model.JumpStartModel, which was removed in V3.)
import uuid
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.train.configs import Compute

unique_id = str(uuid.uuid4())[:8]
pretrained_endpoint_name = f"llama3-8b-pretrained-{unique_id}"

# Please set accept_eula=True to accept the model EULA before deploying.
pretrained_config = JumpStartConfig(
    model_id=model_id,
    model_version=model_version,
    accept_eula=True,
)
pretrained_model_builder = ModelBuilder.from_jumpstart_config(
    jumpstart_config=pretrained_config,
    compute=Compute(instance_type="ml.g5.2xlarge"),
)
pretrained_model = pretrained_model_builder.build()
pretrained_endpoint = pretrained_model_builder.deploy(endpoint_name=pretrained_endpoint_name)
print(f"Pretrained endpoint: {pretrained_endpoint.endpoint_name}")

## Invoke the endpoint

---
Next, we invoke the endpoint with some sample queries. Later, in this notebook, we will fine-tune this model with a custom dataset and carry out inference using the fine-tuned model. We will also show comparison between results obtained via the pre-trained and the fine-tuned models.

---

In [ ]:
def print_response(payload, response):
    print(payload["inputs"])
    print(f"> {response.get('generated_text')}")
    print("\n==================================\n")

In [ ]:
# V3: invoke the endpoint with core Endpoint.invoke instead of predictor.predict.
import json

payload = {
    "inputs": "I believe the meaning of life is",
    "parameters": {
        "max_new_tokens": 64,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}
try:
    # Please change custom_attributes to "accept_eula=true" to run inference.
    result = pretrained_endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        custom_attributes="accept_eula=false",
    )
    response = json.loads(result.body.read().decode("utf-8"))
    print_response(payload, response)
except Exception as e:
    print(e)

---
To learn about additional use cases of pre-trained model, please checkout the notebook [Text completion: Run Llama 3 models in SageMaker JumpStart](https://github.com/aws/amazon-sagemaker-examples/blob/main/introduction_to_amazon_algorithms/jumpstart-foundation-models/llama-3-text-completion.ipynb).

---

## Dataset preparation for fine-tuning

---

You can fine-tune on the dataset with domain adaptation format or instruction tuning format. Please find more details in the section [Dataset instruction](#Dataset-instruction). In this demo, we will use a subset of [Dolly dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k) in an instruction tuning format. Dolly dataset contains roughly 15,000 instruction following records for various categories such as question answering, summarization, information extraction etc. It is available under Apache 2.0 license. We will select the summarization examples for fine-tuning.


Training data is formatted in JSON lines (.jsonl) format, where each line is a dictionary representing a single data sample. All training data must be in a single folder, however it can be saved in multiple jsonl files. The training folder can also contain a template.json file describing the input and output formats.

To train your model on a collection of unstructured dataset (text files), please see the section [Example fine-tuning with Domain-Adaptation dataset format](#Example-fine-tuning-with-Domain-Adaptation-dataset-format) in the Appendix.

---

In [ ]:
from datasets import load_dataset

dolly_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

# To train for question answering/information extraction, you can replace the assertion in next line to example["category"] == "closed_qa"/"information_extraction".
summarization_dataset = dolly_dataset.filter(lambda example: example["category"] == "summarization")
summarization_dataset = summarization_dataset.remove_columns("category")

# We split the dataset into two where test data is used to evaluate at the end.
train_and_test_dataset = summarization_dataset.train_test_split(test_size=0.1)

# Dumping the training data to a local file to be used for training.
train_and_test_dataset["train"].to_json("train.jsonl")

In [ ]:
train_and_test_dataset["train"][0]

---
Next, we create a prompt template for using the data in an instruction / input format for the training job (since we are instruction fine-tuning the model in this example), and also for inferencing the deployed endpoint.

---

In [ ]:
import json

template = {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n",
    "completion": " {response}",
}
with open("template.json", "w") as f:
    json.dump(template, f)

### Upload dataset to S3
---

We will upload the prepared dataset to S3 which will be used for fine-tuning.

---

In [ ]:
# V3: sagemaker.s3.S3Uploader was removed. Upload with Session.upload_data.
import random
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session()
output_bucket = sagemaker_session.default_bucket()
train_data_prefix = "dolly_dataset"

train_data_location = sagemaker_session.upload_data(
    path="train.jsonl", bucket=output_bucket, key_prefix=train_data_prefix
)
sagemaker_session.upload_data(
    path="template.json", bucket=output_bucket, key_prefix=train_data_prefix
)
# Training channel is the S3 folder that holds train.jsonl + template.json.
train_data_channel = f"s3://{output_bucket}/{train_data_prefix}"
print(f"Training data: {train_data_channel}")

## Train the model
---
Next, we fine-tune the LLaMA 3 8B model on the summarization dataset from Dolly. Finetuning scripts are based on scripts provided by [this repo](https://github.com/facebookresearch/llama-recipes/tree/main). To learn more about the fine-tuning scripts, please checkout section [5. Few notes about the fine-tuning method](#5.-Few-notes-about-the-fine-tuning-method). For a list of supported hyper-parameters and their default values, please see section [3. Supported Hyper-parameters for fine-tuning](#3.-Supported-Hyper-parameters-for-fine-tuning).

---

In [ ]:
# V3: fine-tune a JumpStart model with ModelTrainer.from_jumpstart_config.
# (V2 used sagemaker.jumpstart.estimator.JumpStartEstimator, removed in V3.)
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import InputData

finetune_config = JumpStartConfig(
    model_id=model_id,
    model_version=model_version,
    accept_eula=True,  # Please accept the model EULA to fine-tune.
)
model_trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=finetune_config,
    compute=Compute(instance_type="ml.g5.12xlarge"),  # For Llama-3-70b use ml.g5.48xlarge
    # By default instruction tuning is off; set instruction_tuned=True to use it.
    hyperparameters={
        "instruction_tuned": "True",
        "epoch": "1",
        "max_input_length": "1024",
    },
    sagemaker_session=sagemaker_session,
)
model_trainer.train(
    input_data_config=[InputData(channel_name="training", data_source=train_data_channel)]
)

# Get the model artifacts path after training.
training_job = model_trainer._latest_training_job
model_artifacts = training_job.model_artifacts.s3_model_artifacts
print(f"Model artifacts: {model_artifacts}")

In [ ]:
# The fine-tuned model artifact S3 URI.
model_path = model_artifacts
model_path

Studio Kernel Dying issue:  If your studio kernel dies and you lose reference to the estimator object, please see section [6. Studio Kernel Dead/Creating JumpStart Model from the training Job](#6.-Studio-Kernel-Dead/Creating-JumpStart-Model-from-the-training-Job) on how to deploy endpoint using the training job name and the model id. 


### Deploy the fine-tuned model
---
Next, we deploy fine-tuned model. We will compare the performance of fine-tuned and pre-trained model.

---

In [ ]:
# V3: deploy the fine-tuned model by chaining ModelBuilder off the trainer.
finetuned_endpoint_name = f"llama3-8b-finetuned-{unique_id}"
finetuned_model_builder = ModelBuilder(model=model_trainer)
finetuned_model = finetuned_model_builder.build()
finetuned_endpoint = finetuned_model_builder.deploy(endpoint_name=finetuned_endpoint_name)
print(f"Fine-tuned endpoint: {finetuned_endpoint.endpoint_name}")

### Evaluate the pre-trained and fine-tuned model
---
Next, we use the test data to evaluate the performance of the fine-tuned model and compare it with the pre-trained model. 

---

In [ ]:
import pandas as pd
from IPython.display import display, HTML

test_dataset = train_and_test_dataset["test"]

(
    inputs,
    ground_truth_responses,
    responses_before_finetuning,
    responses_after_finetuning,
) = (
    [],
    [],
    [],
    [],
)


def predict_and_print(datapoint):
    # For instruction fine-tuning, we insert a special key between input and output
    input_output_demarkation_key = "\n\n### Response:\n"

    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], context=datapoint["context"]
        )
        + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["response"])
    # Please change custom_attributes to "accept_eula=true" to run inference.
    pretrained_result = pretrained_endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        custom_attributes="accept_eula=false",
    )
    pretrained_response = json.loads(pretrained_result.body.read().decode("utf-8"))
    responses_before_finetuning.append(pretrained_response.get("generated_text"))
    # Fine-tuned Llama 3 models don't require accept_eula=true.
    finetuned_result = finetuned_endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
    )
    finetuned_response = json.loads(finetuned_result.body.read().decode("utf-8"))
    responses_after_finetuning.append(finetuned_response.get("generated_text"))


try:
    for i, datapoint in enumerate(test_dataset.select(range(5))):
        predict_and_print(datapoint)

    df = pd.DataFrame(
        {
            "Inputs": inputs,
            "Ground Truth": ground_truth_responses,
            "Response from non-finetuned model": responses_before_finetuning,
            "Response from fine-tuned model": responses_after_finetuning,
        }
    )
    display(HTML(df.to_html()))
except Exception as e:
    print(e)

### Clean up resources

In [ ]:
# V3: delete model, endpoint, and endpoint config for both endpoints.
from sagemaker.core.resources import EndpointConfig

for _model, _endpoint in [
    (pretrained_model, pretrained_endpoint),
    (finetuned_model, finetuned_endpoint),
]:
    try:
        _endpoint_config = EndpointConfig.get(endpoint_config_name=_endpoint.endpoint_name)
        _model.delete()
        _endpoint.delete()
        _endpoint_config.delete()
    except Exception as e:
        print(e)
print("Pretrained and fine-tuned resources cleaned up.")

### Import the fine tuned model to Jumpstart private model hub
---
Next, we will prepare the model artifact and upload it into the Sagemaker's default S3 bucket 

---

In [ ]:
# V3: the private model hub is managed with sagemaker.core.resources.Hub.
# (V2 used sagemaker.jumpstart.hub.hub.Hub, which was removed in V3.)
import utils
import boto3
from sagemaker.core.helper.session_helper import Session

sess = Session()
default_bucket = sess.default_bucket()
print(default_bucket)

In [ ]:
# download model artifacts
utils.download_model_from_s3(model_path)

# Create the model.tar.gz
print("Creating the model artifact.....")
utils.package_llama_model('downloaded_model', 'model.tar.gz')  # this process take a while
print("Created the model artifact!!!")

In [ ]:
# upload model artifact to S3
# we will upload it to the output folder in model_path variable

model_upload_path = model_path.rsplit('/', 2)[0] + '/'


print(f"Uploading the model artifact to s3: {model_upload_path}")

s3_artifact_uri = utils.upload_model_to_s3(model_upload_path)

In [ ]:
# Define the private model hub details to create one.
HUB_NAME = "Custom-Model-HubZ"
HUB_DISPLAY_NAME = "Custom-Model-HubZ"
REGION = "us-east-1"

sm_client = boto3.client("sagemaker", region_name=REGION)
print(sess.get_caller_identity_arn())

In [ ]:
# V3: create the private hub with the Hub resource class.
from sagemaker.core.resources import Hub

try:
    hub = Hub.create(
        hub_name=HUB_NAME,
        hub_description="This is a Curated Hub. Replace this with a description which explains the purpose of this Hub.",
        hub_display_name=HUB_DISPLAY_NAME,
        region=REGION,
    )
    print(f"Successfully created Hub with name {HUB_NAME} in {REGION}")
except Exception as e:
    if "ResourceInUse" in str(e):
        print(f"A hub with the name {HUB_NAME} already exists in your account.")
        hub = Hub.get(hub_name=HUB_NAME, region=REGION)
    else:
        raise e

In [ ]:
## Retrieve to update the HubContentDocument with the fine-tuned model artifacts

import json
hub_content = json.loads(utils.get_hub_content_document())


In [ ]:
new_hostingartifact_uri = s3_artifact_uri

utils.update_hosting_uris(hub_content, new_hostingartifact_uri)

In [ ]:
# Convert the dictionary to a JSON string
import json
hub_doc = json.dumps(hub_content)

In [ ]:
# importing to the hub (boto3 import_hub_content; no high-level V3 wrapper)
HUB_CONTENT_NAME = f"fine-tuned-llama3-8B-{unique_id}"
sm_client.import_hub_content(
    HubContentName=HUB_CONTENT_NAME,
    HubContentType="Model",
    DocumentSchemaVersion="2.3.0",
    HubName=HUB_NAME,
    HubContentDocument=hub_doc,
)

In [ ]:
# list the models from the private hub (V3 Hub has no list_models; use boto3)
hub_contents = sm_client.list_hub_contents(HubName=HUB_NAME, HubContentType="Model")
hub_contents["HubContentSummaries"]

In [ ]:
# deploy the fine-tuned model from the private hub as an endpoint

# retrieve the hub's ARN
HUB_ARN = hub.hub_arn
print(HUB_ARN)

# find the model we just imported by name (instead of a hardcoded index)
summaries = sm_client.list_hub_contents(HubName=HUB_NAME, HubContentType="Model")[
    "HubContentSummaries"
]
target = next(s for s in summaries if s["HubContentName"] == HUB_CONTENT_NAME)
hub_model_id = target["HubContentName"]
hub_model_version = target["HubContentVersion"]
print(hub_model_id, hub_model_version)

# V3: deploy from the private hub via ModelBuilder.from_jumpstart_config with hub_name.
hub_endpoint_name = f"llama3-8b-from-hub-{unique_id}"
hub_jumpstart_config = JumpStartConfig(
    model_id=hub_model_id,
    model_version=hub_model_version,
    hub_name=HUB_ARN,
    accept_eula=True,
)
hub_model_builder = ModelBuilder.from_jumpstart_config(
    jumpstart_config=hub_jumpstart_config,
    compute=Compute(instance_type="ml.g5.12xlarge"),
)
hub_model = hub_model_builder.build()
hub_endpoint = hub_model_builder.deploy(endpoint_name=hub_endpoint_name)
endpoint_name = hub_endpoint.endpoint_name
endpoint_name

In [ ]:
# invoke the endpoint (V3 core Endpoint.invoke)
payload = {
    "inputs": "Write a poem about the spring season.",
    "parameters": {
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True,
    },
}
try:
    result = hub_endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        custom_attributes="accept_eula=true",  # Required for Llama models
    )
    response_body = json.loads(result.body.read().decode("utf-8"))
    print(response_body)
except Exception as e:
    print(f"Error invoking endpoint: {str(e)}")

### Clean up resources

In [ ]:
# Clean up the hub-deployed endpoint, model, and endpoint config.
try:
    hub_endpoint_config = EndpointConfig.get(endpoint_config_name=endpoint_name)
    hub_model.delete()
    hub_endpoint.delete()
    hub_endpoint_config.delete()
    print("Hub-deployed resources cleaned up.")
except Exception as e:
    print(e)

# Appendix

### 1. Supported Inference Parameters

---
This model supports the following inference payload parameters:

* **max_new_tokens:** Model generates text until the output length (excluding the input context length) reaches max_new_tokens. If specified, it must be a positive integer.
* **temperature:** Controls the randomness in the output. Higher temperature results in output sequence with low-probability words and lower temperature results in output sequence with high-probability words. If `temperature` -> 0, it results in greedy decoding. If specified, it must be a positive float.
* **top_p:** In each step of text generation, sample from the smallest possible set of words with cumulative probability `top_p`. If specified, it must be a float between 0 and 1.
* **return_full_text:** If True, input text will be part of the output generated text. If specified, it must be boolean. The default value for it is False.

You may specify any subset of the parameters mentioned above while invoking an endpoint. 


### Notes
- If `max_new_tokens` is not defined, the model may generate up to the maximum total tokens allowed, which is 8K for these models. This may result in endpoint query timeout errors, so it is recommended to set `max_new_tokens` when possible. For 8B and 70B models, we recommend to set `max_new_tokens` no greater than 1500 and 500 respectively, while keeping the total number of tokens less than 8K.
- In order to support a 8k context length, this model has restricted query payloads to only utilize a batch size of 1. Payloads with larger batch sizes will receive an endpoint error prior to inference.

---

### 2. Dataset formatting instruction for training

---

####  Fine-tune the Model on a New Dataset
We currently offer two types of fine-tuning: instruction fine-tuning and domain adaption fine-tuning. You can easily switch to one of the training 
methods by specifying parameter `instruction_tuned` being 'True' or 'False'.


#### 2.1. Domain adaptation fine-tuning
The Text Generation model can also be fine-tuned on any domain specific dataset. After being fine-tuned on the domain specific dataset, the model
is expected to generate domain specific text and solve various NLP tasks in that specific domain with **few shot prompting**.

Below are the instructions for how the training data should be formatted for input to the model.

- **Input:** A train and an optional validation directory. Each directory contains a CSV/JSON/TXT file. 
  - For CSV/JSON files, the train or validation data is used from the column called 'text' or the first column if no column called 'text' is found.
  - The number of files under train and validation (if provided) should equal to one, respectively. 
- **Output:** A trained model that can be deployed for inference. 

Below is an example of a TXT file for fine-tuning the Text Generation model. The TXT file is SEC filings of Amazon from year 2021 to 2022.

```Note About Forward-Looking Statements
This report includes estimates, projections, statements relating to our
business plans, objectives, and expected operating results that are “forward-
looking statements” within the meaning of the Private Securities Litigation
Reform Act of 1995, Section 27A of the Securities Act of 1933, and Section 21E
of the Securities Exchange Act of 1934. Forward-looking statements may appear
throughout this report, including the following sections: “Business” (Part I,
Item 1 of this Form 10-K), “Risk Factors” (Part I, Item 1A of this Form 10-K),
and “Management’s Discussion and Analysis of Financial Condition and Results
of Operations” (Part II, Item 7 of this Form 10-K). These forward-looking
statements generally are identified by the words “believe,” “project,”
“expect,” “anticipate,” “estimate,” “intend,” “strategy,” “future,”
“opportunity,” “plan,” “may,” “should,” “will,” “would,” “will be,” “will
continue,” “will likely result,” and similar expressions. Forward-looking
statements are based on current expectations and assumptions that are subject
to risks and uncertainties that may cause actual results to differ materially.
We describe risks and uncertainties that could cause actual results and events
to differ materially in “Risk Factors,” “Management’s Discussion and Analysis
of Financial Condition and Results of Operations,” and “Quantitative and
Qualitative Disclosures about Market Risk” (Part II, Item 7A of this Form
10-K). Readers are cautioned not to place undue reliance on forward-looking
statements, which speak only as of the date they are made. We undertake no
obligation to update or revise publicly any forward-looking statements,
whether because of new information, future events, or otherwise.
GENERAL
Embracing Our Future ...
```


#### 2.2. Instruction fine-tuning
The Text generation model can be instruction-tuned on any text data provided that the data 
is in the expected format. The instruction-tuned model can be further deployed for inference. 
Below are the instructions for how the training data should be formatted for input to the 
model.

Below are the instructions for how the training data should be formatted for input to the model.

- **Input:** A train and an optional validation directory. Train and validation directories should contain one or multiple JSON lines (`.jsonl`) formatted files. In particular, train directory can also contain an optional `*.json` file describing the input and output formats. 
  - The best model is selected according to the validation loss, calculated at the end of each epoch.
  If a validation set is not given, an (adjustable) percentage of the training data is
  automatically split and used for validation.
  - The training data must be formatted in a JSON lines (`.jsonl`) format, where each line is a dictionary
representing a single data sample. All training data must be in a single folder, however
it can be saved in multiple jsonl files. The `.jsonl` file extension is mandatory. The training
folder can also contain a `template.json` file describing the input and output formats. If no
template file is given, the following template will be used:
  ```json
  {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Input:\n{context}",
    "completion": "{response}"
  }
  ```
  - In this case, the data in the JSON lines entries must include `instruction`, `context` and `response` fields. If a custom template is provided it must also use `prompt` and `completion` keys to define
  the input and output templates.
  Below is a sample custom template:

  ```json
  {
    "prompt": "question: {question} context: {context}",
    "completion": "{answer}"
  }
  ```
Here, the data in the JSON lines entries must include `question`, `context` and `answer` fields. 
- **Output:** A trained model that can be deployed for inference. 

---

#### 2.3. Example fine-tuning with Domain-Adaptation dataset format
---
We provide a subset of SEC filings data of Amazon in domain adaptation dataset format. It is downloaded from publicly available [EDGAR](https://www.sec.gov/edgar/searchedgar/companysearch). Instruction of accessing the data is shown [here](https://www.sec.gov/os/accessing-edgar-data).

License: [Creative Commons Attribution-ShareAlike License (CC BY-SA 4.0)](https://creativecommons.org/licenses/by-sa/4.0/legalcode).

Please uncomment the following code to fine-tune the model on dataset in domain adaptation format.

---

In [ ]:
# Appendix example: domain-adaptation fine-tuning with ModelTrainer (V3).
# This is an optional illustrative example. Uncomment the code below to run a
# domain-adaptation fine-tuning job on the sample SEC filings dataset.
#
# import boto3
#
# appendix_model_id = "meta-textgeneration-llama-3-8b"
# region_name = boto3.Session().region_name
#
# appendix_config = JumpStartConfig(
#     model_id=appendix_model_id,
#     accept_eula=True,
# )
# appendix_trainer = ModelTrainer.from_jumpstart_config(
#     jumpstart_config=appendix_config,
#     compute=Compute(instance_type="ml.g5.24xlarge"),
#     hyperparameters={"instruction_tuned": "False", "epoch": "5"},
# )
# appendix_trainer.train(
#     input_data_config=[
#         InputData(
#             channel_name="training",
#             data_source=f"s3://jumpstart-cache-prod-{region_name}/training-datasets/sec_amazon",
#         )
#     ]
# )

### 3. Supported Hyper-parameters for fine-tuning
---
- epoch: The number of passes that the fine-tuning algorithm takes through the training dataset. Must be an integer greater than 1. Default: 5
- learning_rate: The rate at which the model weights are updated after working through each batch of training examples. Must be a positive float greater than 0. Default: 1e-4.
- instruction_tuned: Whether to instruction-train the model or not. Must be 'True' or 'False'. Default: 'False'
- per_device_train_batch_size: The batch size per GPU core/CPU for training. Must be a positive integer. Default: 4.
- per_device_eval_batch_size: The batch size per GPU core/CPU for evaluation. Must be a positive integer. Default: 1
- max_train_samples: For debugging purposes or quicker training, truncate the number of training examples to this value. Value -1 means using all of training samples. Must be a positive integer or -1. Default: -1. 
- max_val_samples: For debugging purposes or quicker training, truncate the number of validation examples to this value. Value -1 means using all of validation samples. Must be a positive integer or -1. Default: -1. 
- max_input_length: Maximum total input sequence length after tokenization. Sequences longer than this will be truncated. If -1, max_input_length is set to the minimum of 1024 and the maximum model length defined by the tokenizer. If set to a positive value, max_input_length is set to the minimum of the provided value and the model_max_length defined by the tokenizer. Must be a positive integer or -1. Default: -1. 
- validation_split_ratio: If validation channel is none, ratio of train-validation split from the train data. Must be between 0 and 1. Default: 0.2. 
- train_data_split_seed: If validation data is not present, this fixes the random splitting of the input training data to training and validation data used by the algorithm. Must be an integer. Default: 0.
- preprocessing_num_workers: The number of processes to use for the preprocessing. If None, main process is used for preprocessing. Default: "None"
- lora_r: Lora R. Must be a positive integer. Default: 8.
- lora_alpha: Lora Alpha. Must be a positive integer. Default: 32
- lora_dropout: Lora Dropout. must be a positive float between 0 and 1. Default: 0.05. 
- int8_quantization: If True, model is loaded with 8 bit precision for training. Default for 8B: False. Default for 70B: True.
- enable_fsdp: If True, training uses Fully Sharded Data Parallelism. Default for 8B: True. Default for 70B: False.

Note 1: int8_quantization is not supported with FSDP. Also, int8_quantization = 'False' and enable_fsdp = 'False' is not supported due to CUDA memory issues for any of the g5 family instances. Thus, we recommend setting exactly one of int8_quantization or enable_fsdp to be 'True'
Note 2: Due to the size of the model, 70B model can not be fine-tuned with enable_fsdp = 'True' for any of the supported instance types.

---

### 4. Supported Instance types for fine-tuning Llama 3

---
We have tested our scripts on the following instances types for fine-tuning Llama 3:

| Model | Model ID | All Supported Instances Types for fine-tuning |
| - | - | - |
| Llama 3 8B | meta-textgeneration-llama-3-8b | ml.g5.12xlarge, ml.g5.24xlarge, ml.g5.48xlarge, ml.p3dn.24xlarge, ml.g4dn.12xlarge |
| Llama 3 8B Instruct | meta-textgeneration-llama-3-8b-instruct | ml.g5.12xlarge, ml.g5.24xlarge, ml.g5.48xlarge, ml.p3dn.24xlarge, ml.g4dn.12xlarge  |
| Llama 3 70B | meta-textgeneration-llama-3-70b | ml.g5.48xlarge, ml.p4d.24xlarge |
| Llama 3 70B Instruct | meta-textgeneration-llama-3-70b-instruct | ml.g5.48xlarge, ml.p4d.24xlarge |

Other instance types may also work to fine-tune. Note: When using p3 instances, training will be done with 32 bit precision as bfloat16 is not supported on these instances. Thus, training job would consume double the amount of CUDA memory when training on p3 instances compared to g5 instances.

---

### 5. Few notes about the fine-tuning method

---
- Fine-tuning scripts are based on [this repo](https://github.com/facebookresearch/llama-recipes/tree/main). 
- Instruction tuning dataset is first converted into domain adaptation dataset format before fine-tuning. 
- Fine-tuning scripts utilize Fully Sharded Data Parallel (FSDP) as well as Low Rank Adaptation (LoRA) method fine-tuning the models

---

### 6. Studio Kernel Dead/Creating JumpStart Model from the training Job
---
Due to the size of the Llama 70B model, training job may take several hours and the studio kernel may die during the training phase. However, during this time, training is still running in SageMaker. If this happens, you can still deploy the endpoint using the training job name with the following code:

How to find the training job name? Go to Console -> SageMaker -> Training -> Training Jobs -> Identify the training job name and substitute in the following cell. 

---

In [ ]:
# NOTE (V3): the V2 JumpStartEstimator.attach(training_job_name) helper, used to
# re-create an estimator from a running/finished training job, is not available in
# V3. If your kernel dies during training, the training job keeps running in
# SageMaker. To deploy afterwards, look up the completed training job and build a
# ModelBuilder from its TrainingJob resource, then deploy. Example (illustrative):
#
#   from sagemaker.core.resources import TrainingJob
#   from sagemaker.serve.model_builder import ModelBuilder
#   training_job = TrainingJob.get(training_job_name="<training-job-name>")
#   model_builder = ModelBuilder(model=training_job)
#   model_builder.build()
#   model_builder.deploy(endpoint_name="<endpoint-name>")
#
# This cell is reference-only and intentionally performs no action.

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)